# EXPEDIA Demo — Phase 4: GenomeOcean Embedding

**Purpose:** Embed the deduplicated marine sequences (produced by Phases 1–3) using GenomeOcean on this Colab GPU. Outputs are written to the project directory — either mounted via Google Drive or kept in `/content/` for the session.

**Flow:**
1. GPU check
2. Point the pipeline at the project directory
3. Install dependencies + load pipeline code
4. Apply demo config
5. Load GenomeOcean
6. Run embedding with auto-resume
7. Verify outputs

**Expected time:** ~2–4 h for 2M sequences on T4 · ~45 min on A100

---
⚠️ **Before starting:** `Runtime → Change runtime type → GPU`

## Cell 1 — GPU check

## Cell 2 — Set project directory

All data lives in one place: `EXPEDIA_Data/` inside the project directory.
Set `PROJECT_DIR` to wherever your pipeline code + data folder live.

**Option A — Google Drive** (data persists across sessions): mount Drive first,
then point `PROJECT_DIR` at the folder containing your code.

**Option B — Colab local `/content/`** (data lost on disconnect): fastest I/O,
fine for a demo where you re-upload or regenerate data each session.

In [ ]:
import os

if FASTA_PATH.exists():
    size_gb = FASTA_PATH.stat().st_size / 1e9
    print(f'FASTA already present: {FASTA_PATH}  ({size_gb:.2f} GB)')
    print('Proceeding to embedding.')
else:
    print('FASTA not found. Upload or copy it to:')
    print(f'  {FASTA_PATH}')
    print()
    print('Option A — Colab file uploader:')
    print('  from google.colab import files')
    print('  uploaded = files.upload()  # pick representative_marine_demo.fasta')
    print('  import shutil')
    print('  shutil.move(list(uploaded.keys())[0], str(FASTA_PATH))')
    print()
    print('Option B — copy from another path already on this machine:')
    print('  import shutil')
    print('  shutil.copy("/path/to/representative_marine_demo.fasta", str(FASTA_PATH))')


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path

DRIVE_ROOT  = Path('/content/drive/MyDrive/EXPEDIA_Data')
EMBED_DIR   = DRIVE_ROOT / '04_embeddings'
CKPT_DIR    = DRIVE_ROOT / 'checkpoints'
MODEL_DIR   = DRIVE_ROOT / 'resources' / 'models' / 'GenomeOcean-4B'
FASTA_DIR   = DRIVE_ROOT / '02_dedup_fasta'

for d in [EMBED_DIR, CKPT_DIR, MODEL_DIR, FASTA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Drive mounted. Directory layout:")
for d in [DRIVE_ROOT, EMBED_DIR, CKPT_DIR, MODEL_DIR, FASTA_DIR]:
    exists = '✓' if d.exists() else '✗'
    print(f"  {exists}  {d}")

## Cell 3 — Upload FASTA from USB

Two options — run whichever applies:

**Option A:** Upload directly from your USB drive using Colab's file uploader  
**Option B:** If you already have the FASTA on Drive from a previous session, skip this cell

In [ ]:
import os

FASTA_PATH = FASTA_DIR / 'representative_marine_demo.fasta'

if FASTA_PATH.exists():
    size_gb = FASTA_PATH.stat().st_size / 1e9
    print(f"FASTA already on Drive: {FASTA_PATH}  ({size_gb:.2f} GB)")
    print("Skipping upload — proceeding to embedding.")
else:
    print("FASTA not found on Drive. Choose an upload method:\n")
    print("Option A — Colab file uploader (small files < 2 GB):")
    print("  from google.colab import files")
    print("  uploaded = files.upload()")
    print("  # then move the file:")
    print("  import shutil")
    print("  shutil.move(list(uploaded.keys())[0], str(FASTA_PATH))")
    print()
    print("Option B — Direct copy if file is already accessible at a Colab path:")
    print("  import shutil")
    print("  shutil.copy('/path/to/representative_marine_demo.fasta', str(FASTA_PATH))")
    print()
    print("Option C — Stream download from a public URL:")
    print("  !wget -O '{FASTA_PATH}' 'https://your-url/representative_marine_demo.fasta'")
    print()
    print("Run one of the above, then re-run this cell to confirm the file is present.")

In [ ]:
# Verify FASTA and count records
if not FASTA_PATH.exists():
    raise FileNotFoundError(
        f"FASTA file not found at {FASTA_PATH}.\n"
        "Complete the upload step in the cell above before continuing."
    )

print(f"FASTA path  : {FASTA_PATH}")
print(f"File size   : {FASTA_PATH.stat().st_size / 1e9:.3f} GB")

# Count records efficiently
n_records = sum(
    chunk.count(b'>')
    for chunk in iter(lambda: FASTA_PATH.open('rb').read(1 << 20), b'')
)

# Faster count using a generator
n_records = 0
with FASTA_PATH.open('rb') as fh:
    for chunk in iter(lambda: fh.read(4 * 1024 * 1024), b''):
        n_records += chunk.count(b'>')

print(f"Records     : {n_records:,}")

## Cell 4 — Install dependencies

In [ ]:
# Only install what isn't already present — avoids re-installing on reconnect
import importlib

def _need(pkg):
    try:
        importlib.import_module(pkg)
        return False
    except ImportError:
        return True

pkgs_needed = []
if _need('transformers'):   pkgs_needed.append('transformers>=4.40')
if _need('accelerate'):     pkgs_needed.append('accelerate')
if _need('sentencepiece'):  pkgs_needed.append('sentencepiece')  # BPE tokeniser
if _need('einops'):         pkgs_needed.append('einops')          # GenomeOcean dep

if pkgs_needed:
    print(f"Installing: {pkgs_needed}")
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + pkgs_needed
    )
    print("Installation complete.")
else:
    print("All dependencies already installed.")

## Cell 5 — Clone or upload pipeline code

In [ ]:
# Add the pipeline code to sys.path so phase4_embedding and its imports resolve.
# Two options depending on how you want to manage code:
#
# Option A (recommended for demo): copy pipeline files from Drive
#   — Upload the expedia_tier2/ folder to EXPEDIA_Data/code/ on Drive once,
#     then this cell copies it to /content/ every session.
#
# Option B: clone from your GitHub repo
#   !git clone https://github.com/FaisalTabrez/DeepBio_Edge_v4 /content/expedia_tier2

import shutil

CODE_SRC  = DRIVE_ROOT / 'code' / 'expedia_tier2'   # upload once to Drive
CODE_DEST = Path('/content/expedia_tier2')

if CODE_SRC.exists():
    if CODE_DEST.exists():
        shutil.rmtree(str(CODE_DEST))
    shutil.copytree(str(CODE_SRC), str(CODE_DEST))
    print(f"Pipeline code copied from Drive → {CODE_DEST}")
elif CODE_DEST.exists():
    print(f"Pipeline code already at {CODE_DEST} (from previous cell / manual upload)")
else:
    print(
        "Pipeline code not found.\n"
        f"  Either upload expedia_tier2/ to {CODE_SRC} on Drive,\n"
        "  or manually upload the files using the Files panel on the left,\n"
        "  or clone from GitHub (see comment above)."
    )

if str(CODE_DEST) not in sys.path:
    sys.path.insert(0, str(CODE_DEST))
    print(f"Added {CODE_DEST} to sys.path")

## Cell 6 — Apply demo configuration

In [ ]:
import os, sys

# EXPEDIA_ROOT and EXPEDIA_DEMO are already set in Cell 2.
# Just apply the demo overlay and confirm the active paths.

import pipeline_config          # must import before demo_config
from demo_config import apply_demo_config
apply_demo_config()

import pipeline_config as cfg
print('Active Phase 4 configuration:')
print(f'  FASTA input     : {FASTA_PATH}')
print(f'  Embed output    : {cfg.EMBED_OUTPUT_NPY}')
print(f'  IDs output      : {cfg.EMBED_IDS_TXT}')
print(f'  Checkpoint file : {cfg.EMBED_CHECKPOINT}')
print(f'  Model cache     : {cfg.GENOME_OCEAN_LOCAL_DIR}')
print(f'  Batch size      : {cfg.EMBED_BATCH_SIZE}')
print(f'  Checkpoint N    : {cfg.EMBED_CHECKPOINT_N:,}')


## Cell 7 — Load GenomeOcean model

First run: pulls ~8 GB from HuggingFace and caches to Drive.  
Subsequent runs: loads directly from the Drive cache (no download).

In [ ]:
import time

from phase4_embedding import load_genome_ocean

print("Loading GenomeOcean …")
t0 = time.monotonic()

model, tokenizer, device = load_genome_ocean(local_dir=cfg.GENOME_OCEAN_LOCAL_DIR)

elapsed = time.monotonic() - t0
print(f"Model loaded in {elapsed:.1f} s on {device}")

# Sanity-check: embed one dummy sequence
import numpy as np
from phase4_embedding import _embed_batch

dummy = _embed_batch(['ATCGATCGATCG'], model, tokenizer, device)
assert dummy.shape == (1, cfg.EMBED_DIMENSION), f"Expected (1, {cfg.EMBED_DIMENSION}), got {dummy.shape}"
print(f"Embedding sanity check passed: shape={dummy.shape}  dtype={dummy.dtype}")

## Cell 8 — Run embedding with live progress

This cell is the main compute step.  It is safe to interrupt and re-run — the resume guard reads the checkpoint file from Drive and skips already-embedded sequences.

In [ ]:
import json
import time
from IPython.display import clear_output, display
from phase4_embedding import (
    _count_fasta_records,
    _iter_fasta,
    _embed_batch,
    _flush_embeddings,
)

# ── Resume guard ────────────────────────────────────────────────────────────
CKPT_FILE = cfg.EMBED_CHECKPOINT
cp = json.loads(CKPT_FILE.read_text()) if CKPT_FILE.exists() else {}

if cp.get('done'):
    print(f"Embedding already complete ({cp['n_embedded']:,} sequences). "
          "Nothing to do.\nDelete the checkpoint file to re-embed from scratch:")
    print(f"  {CKPT_FILE}")
else:
    start_from = cp.get('last_index', 0)
    if start_from > 0:
        print(f"Resuming from sequence index {start_from:,}")
    else:
        print("Starting fresh embedding run.")

    # ── Count total sequences ─────────────────────────────────────────────
    print("Counting sequences …")
    n_total = _count_fasta_records(FASTA_PATH)
    print(f"Total sequences: {n_total:,}")

    # ── Embedding loop with progress tracking ────────────────────────────
    batch_accs  = []
    batch_seqs  = []
    acc_chunks  = []
    vec_chunks  = []
    processed   = start_from
    t_start     = time.monotonic()
    t_last_ckpt = t_start

    ids_mode = 'a' if start_from > 0 else 'w'

    with cfg.EMBED_IDS_TXT.open(ids_mode, encoding='utf-8') as id_fh:
        for acc, seq in _iter_fasta(FASTA_PATH, start_idx=start_from):
            batch_accs.append(acc)
            batch_seqs.append(seq)

            if len(batch_seqs) >= cfg.EMBED_BATCH_SIZE:
                vecs = _embed_batch(batch_seqs, model, tokenizer, device)
                vec_chunks.append(vecs)
                acc_chunks.extend(batch_accs)
                id_fh.write('\n'.join(batch_accs) + '\n')
                processed += len(batch_accs)
                batch_accs, batch_seqs = [], []

                # ── Checkpoint every EMBED_CHECKPOINT_N sequences ────────
                if processed % cfg.EMBED_CHECKPOINT_N < cfg.EMBED_BATCH_SIZE:
                    _flush_embeddings(vec_chunks, cfg.EMBED_OUTPUT_NPY, processed, start_from)
                    vec_chunks = []

                    cp['last_index'] = processed
                    CKPT_FILE.write_text(json.dumps(cp, indent=2))
                    t_last_ckpt = time.monotonic()

                # ── Live progress display (updates every batch) ───────────
                elapsed   = time.monotonic() - t_start
                done_frac = (processed - start_from) / max(n_total - start_from, 1)
                rate      = (processed - start_from) / max(elapsed, 1)
                eta_s     = (n_total - processed) / max(rate, 1)
                eta_min   = eta_s / 60
                bar_fill  = int(done_frac * 40)
                bar       = '█' * bar_fill + '░' * (40 - bar_fill)

                clear_output(wait=True)
                print(f"EXPEDIA Demo — Phase 4 Embedding")
                print(f"{'─'*50}")
                print(f"Progress  [{bar}]  {done_frac*100:.1f}%")
                print(f"Processed : {processed:>10,} / {n_total:,}")
                print(f"Speed     : {rate:>10,.0f} seq/s")
                print(f"ETA       : {eta_min:>10.1f} min")
                print(f"Elapsed   : {elapsed/60:>10.1f} min")
                print(f"Last ckpt : {(time.monotonic()-t_last_ckpt)/60:.1f} min ago")
                print(f"Output    : {cfg.EMBED_OUTPUT_NPY.name}")

        # ── Flush final partial batch ─────────────────────────────────────
        if batch_seqs:
            vecs = _embed_batch(batch_seqs, model, tokenizer, device)
            vec_chunks.append(vecs)
            id_fh.write('\n'.join(batch_accs) + '\n')
            processed += len(batch_accs)

        if vec_chunks:
            _flush_embeddings(vec_chunks, cfg.EMBED_OUTPUT_NPY, processed, start_from)

    # ── Mark complete ─────────────────────────────────────────────────────
    cp['done']       = True
    cp['n_embedded'] = processed
    cp['embed_dim']  = cfg.EMBED_DIMENSION
    CKPT_FILE.write_text(json.dumps(cp, indent=2))

    total_min = (time.monotonic() - t_start) / 60
    clear_output(wait=True)
    print(f"✓  Embedding complete!")
    print(f"   Sequences : {processed:,}")
    print(f"   Time      : {total_min:.1f} min")
    print(f"   Output    : {cfg.EMBED_OUTPUT_NPY}")
    print(f"   IDs file  : {cfg.EMBED_IDS_TXT}")

## Cell 9 — Verify outputs

In [ ]:
import numpy as np

npy_path = cfg.EMBED_OUTPUT_NPY
ids_path = cfg.EMBED_IDS_TXT

if not npy_path.exists():
    print("Embeddings file not found — has Cell 8 run to completion?")
else:
    embeddings    = np.load(npy_path, mmap_mode='r')
    accession_ids = [l.strip() for l in ids_path.read_text().splitlines() if l.strip()]

    print("Output verification")
    print('─' * 40)
    print(f"Embeddings shape : {embeddings.shape}")
    print(f"Embeddings dtype : {embeddings.dtype}")
    print(f"Accession IDs    : {len(accession_ids):,}")
    print(f"File size        : {npy_path.stat().st_size / 1e9:.3f} GB")
    print()

    # Shape sanity check
    assert embeddings.shape[0] == len(accession_ids), (
        f"Mismatch: {embeddings.shape[0]} vectors vs {len(accession_ids)} IDs"
    )
    assert embeddings.shape[1] == cfg.EMBED_DIMENSION, (
        f"Wrong dimension: {embeddings.shape[1]} (expected {cfg.EMBED_DIMENSION})"
    )

    # Norm distribution (L2 norms should be roughly 1–20 before normalisation)
    sample = embeddings[:1000]
    norms  = np.linalg.norm(sample, axis=1)
    print(f"L2 norm (sample) — mean: {norms.mean():.2f}  std: {norms.std():.2f}  "
          f"min: {norms.min():.2f}  max: {norms.max():.2f}")

    # NaN/Inf check
    n_nan = int(np.isnan(embeddings).sum())
    n_inf = int(np.isinf(embeddings).sum())
    if n_nan == 0 and n_inf == 0:
        print("No NaN or Inf values detected — embeddings look clean.")
    else:
        print(f"WARNING: {n_nan} NaN and {n_inf} Inf values found.")
        print("These sequences may need to be re-embedded or filtered.")

    print()
    print(f"First 5 accessions: {accession_ids[:5]}")
    print()
    print("✓  All checks passed. Embeddings ready for Phase 5 (indexing).")

## Cell 10 — What to do after embedding completes

All outputs are already in the project directory — no copying needed.

Run Phases 5–7 locally (or in another Colab session) by pointing the
orchestrator at the same `EXPEDIA_Data/` folder:

```bash
# In your terminal, from the project root:
EXPEDIA_DEMO=1 python orchestrator.py --phases 5 6 7
```

If the data lives on Drive and you want a local copy:

```bash
# Mount Drive with rclone, then just run phases 5-7 against the mounted path:
rclone mount gdrive:expedia_tier2/EXPEDIA_Data ~/expedia_data --daemon
EXPEDIA_ROOT=~/expedia_data EXPEDIA_DEMO=1 python orchestrator.py --phases 5 6 7
```


## Cell 11 — Session reconnect helper

If your Colab session disconnects mid-embedding, run cells 1 → 7 → 8 in sequence.  
Cell 8 will automatically detect the checkpoint on Drive and resume from where it left off.  
The cell below is a one-click shortcut that replays that sequence.

In [ ]:
# ── Quick-resume block (run this after a disconnect) ────────────────────────
import json, os
from pathlib import Path

# Re-apply env-vars in case the session restarted
os.environ['EXPEDIA_ROOT'] = str(DATA_DIR)  # DATA_DIR set in Cell 2
os.environ['EXPEDIA_DEMO'] = '1'

CKPT_FILE = DATA_DIR / 'checkpoints' / 'phase4_embedding.json'

if CKPT_FILE.exists():
    cp = json.loads(CKPT_FILE.read_text())
    if cp.get('done'):
        print(f"Embedding already complete: {cp['n_embedded']:,} sequences. Nothing to do.")
    else:
        last = cp.get('last_index', 0)
        print(f'Checkpoint found — will resume from sequence {last:,}.')
        print('Run Cell 8 (the main embedding loop) to continue.')
else:
    print('No checkpoint found. Run from Cell 4 for a fresh start.')
